### 🧬 网络结构与符号定义

首先，我们明确网络结构和计算过程中使用的符号：

*   **输入层:** 输入向量为 `x`。
*   **隐藏层:**
    *   权重矩阵为 `W₁`，偏置向量为 `b₁`。
    *   线性变换结果（预激活值）为 `z₁ = W₁x + b₁`。
    *   激活函数为 Sigmoid，即 `σ(z) = 1 / (1 + e⁻ᶻ)`。
    *   隐藏层输出（激活值）为 `a₁ = σ(z₁)`。
*   **输出层:**
    *   权重矩阵为 `W₂`，偏置为 `b₂`。
    *   线性变换结果为 `z₂ = W₂a₁ + b₂`。
    *   输出层激活值（即模型预测值）为 `ŷ = a₂ = σ(z₂)`。
*   **损失函数:**
    *   我们使用均方误差（MSE），对于单个样本 `(x, y)`，损失 `L` 为：
        `L = ½(ŷ - y)²`

我们的目标是计算损失 `L` 对所有参数 `W₂, b₂, W₁, b₁` 的梯度，即 `∂L/∂W₂`, `∂L/∂b₂`, `∂L/∂W₁`, `∂L/∂b₁`。

---

### 🚀 第一步：前向传播 (Forward Pass)

前向传播是按照网络结构，从输入到输出依次计算每一层的值。这个过程会保存所有中间结果（`z₁, a₁, z₂, a₂`），这些结果在反向传播中会被反复使用。

1.  `z₁ = W₁x + b₁`
2.  `a₁ = σ(z₁)`
3.  `z₂ = W₂a₁ + b₂`
4.  `ŷ = a₂ = σ(z₂)`
5.  `L = ½(ŷ - y)²`

---

### 🔙 第二步：反向传播 (Backward Pass) - 链式法则推导

反向传播从输出层开始，利用链式法则，逐层向后计算梯度。

#### 1. 计算输出层的梯度

我们首先计算损失 `L` 相对于输出层参数 `W₂` 和 `b₂` 的梯度。根据链式法则，我们需要先计算损失相对于 `z₂` 的梯度，我们将其定义为输出层的“误差项” `δ₂`。

*   **计算误差项 `δ₂ = ∂L/∂z₂`**

    应用链式法则：
    `δ₂ = ∂L/∂z₂ = (∂L/∂ŷ) ⋅ (∂ŷ/∂z₂)`

    我们来分别计算这两部分：
    *   `∂L/∂ŷ = ∂(½(ŷ - y)²)/∂ŷ = (ŷ - y)`
    *   `∂ŷ/∂z₂ = ∂σ(z₂)/∂z₂`。Sigmoid 函数的导数有一个非常好的性质：`σ'(z) = σ(z)(1 - σ(z))`。因此，`∂ŷ/∂z₂ = ŷ(1 - ŷ)`

    将两部分相乘，得到：
    `δ₂ = (ŷ - y) ⋅ ŷ(1 - ŷ)`

*   **计算 `∂L/∂W₂` 和 `∂L/∂b₂`**

    现在我们有了 `δ₂`，就可以计算输出层参数的梯度了。
    *   `∂L/∂W₂ = (∂L/∂z₂) ⋅ (∂z₂/∂W₂) = δ₂ ⋅ (∂(W₂a₁ + b₂)/∂W₂) = δ₂ ⋅ a₁ᵀ`
    *   `∂L/∂b₂ = (∂L/∂z₂) ⋅ (∂z₂/∂b₂) = δ₂ ⋅ (∂(W₂a₁ + b₂)/∂b₂) = δ₂`

    这里 `a₁ᵀ` 是 `a₁` 的转置。

#### 2. 计算隐藏层的梯度

接下来，我们将误差从输出层“反向传播”到隐藏层，计算损失 `L` 相对于隐藏层参数 `W₁` 和 `b₁` 的梯度。同样，我们先计算损失相对于 `z₁` 的梯度，定义为隐藏层的“误差项” `δ₁`。

*   **计算误差项 `δ₁ = ∂L/∂z₁`**

    这是链式法则最核心的体现。`L` 通过 `z₂` 依赖于 `a₁`，而 `a₁` 又通过 `z₁` 依赖于 `W₁` 和 `b₁`。因此：
    `δ₁ = ∂L/∂z₁ = (∂L/∂z₂) ⋅ (∂z₂/∂a₁) ⋅ (∂a₁/∂z₁)`

    我们来逐项分析：
    *   `∂L/∂z₂`：这正是我们刚刚计算出的输出层误差项 `δ₂`。
    *   `∂z₂/∂a₁`：因为 `z₂ = W₂a₁ + b₂`，所以 `∂z₂/∂a₁ = W₂`。
    *   `∂a₁/∂z₁`：因为 `a₁ = σ(z₁)`，所以 `∂a₁/∂z₁ = a₁(1 - a₁)`（Sigmoid 导数）。

    将它们组合起来，得到隐藏层的误差项：
    `δ₁ = (δ₂ ⋅ W₂) ⊙ (a₁(1 - a₁))`

    这里的 `⊙` 表示**逐元素乘法（Hadamard积）**，因为 `δ₂ ⋅ W₂` 和 `a₁(1 - a₁)` 都是向量。

*   **计算 `∂L/∂W₁` 和 `∂L/∂b₁`**

    现在我们有了隐藏层的误差项 `δ₁`，就可以计算隐藏层参数的梯度了。
    *   `∂L/∂W₁ = (∂L/∂z₁) ⋅ (∂z₁/∂W₁) = δ₁ ⋅ (∂(W₁x + b₁)/∂W₁) = δ₁ ⋅ xᵀ`
    *   `∂L/∂b₁ = (∂L/∂z₁) ⋅ (∂z₁/∂b₁) = δ₁ ⋅ (∂(W₁x + b₁)/∂b₁) = δ₁`

---

### 🔄 第三步：参数更新 (Parameter Update)

一旦我们计算出了所有参数的梯度，就可以使用梯度下降法来更新它们，以减小损失。假设学习率为 `η` (eta)，更新规则如下：

*   `W₂ = W₂ - η ⋅ ∂L/∂W₂`
*   `b₂ = b₂ - η ⋅ ∂L/∂b₂`
*   `W₁ = W₁ - η ⋅ ∂L/∂W₁`
*   `b₁ = b₁ - η ⋅ ∂L/∂b₁`

这个过程会针对训练数据集中的每一个（或每一批）样本重复进行，直到模型的损失收敛到一个满意的水平。

### 📌 总结

反向传播的精髓在于**误差项 `δ`** 的递归计算。

*   对于输出层 `L`，`δ_L = ∇ₐL ⊙ σ'(z_L)`
*   对于任意隐藏层 `l`，`δ_l = ((W_{l+1})ᵀ δ_{l+1}) ⊙ σ'(z_l)`

通过这种方式，误差信号 `δ` 像波浪一样从网络的末端向始端传播，每一层都利用后一层传回来的误差和自身的导数，计算出自己对总误差的贡献，并继续向后传递。这种高效的计算方式避免了为每个参数单独求导的巨大开销，是深度学习得以实现的基石。